# U1-04 · Generalizar, pocas etiquetas, vecinos y regresión

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 1, apartados 1.3 a 1.7*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). Las celdas son independientes.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges, con licencia Creative Commons Reconocimiento-CompartirIgual 3.0.

## 1. Generalizar: ni poco ni demasiado (1.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
f = lambda x: np.sin(2 * np.pi * x)        # la regla oculta
x_ent = rng.random(15);  y_ent = f(x_ent) + rng.normal(0, .2, 15)
x_pru = rng.random(200); y_pru = f(x_pru) + rng.normal(0, .2, 200)
ecm = lambda c, x, y: np.mean((np.polyval(c, x) - y) ** 2)

fig, ejes = plt.subplots(1, 3, figsize=(4.4, 1.5), sharey=True)
for g in (1, 2, 3, 5, 8, 10, 12):
    c = np.polyfit(x_ent, y_ent, g)
    print(f"grado {g:2d}: error entrenamiento "
          f"{ecm(c, x_ent, y_ent):.3f}, "
          f"prueba {ecm(c, x_pru, y_pru):.3f}")
    if g in (1, 3, 12):
        eje = ejes[[1, 3, 12].index(g)]
        xs = np.linspace(0, 1, 300)
        eje.plot(xs, np.polyval(c, xs))
        eje.plot(x_ent, y_ent, "o", ms=2)
        eje.set_ylim(-1.8, 1.8); eje.set_title(f"grado {g}")
plt.show()  # fig.savefig("generalizar.pdf")

**Prueba.** Pasa de 15 a 100 puntos de entrenamiento. ¿Sigue disparándose el error del grado 12?

## 2. Agrupar sin etiquetas (1.4)

In [ ]:
import numpy as np
import keras
from sklearn.cluster import KMeans

_, (x, y) = keras.datasets.mnist.load_data()  # 10 000 imágenes
X = x.reshape(-1, 784) / 255.0

# no supervisado: se dan las imágenes, pero no las etiquetas
km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(X)
aciertos, vistos = 0, set()
for g in range(10):
    en_grupo = y[km.labels_ == g]
    mayoritario = np.bincount(en_grupo).argmax()
    aciertos += (en_grupo == mayoritario).sum()
    vistos.add(mayoritario)
    print(f"grupo {g}: {len(en_grupo):5d} imágenes, "
          f"{np.mean(en_grupo == mayoritario):.0%} "
          f"son un {mayoritario}")
print("dígitos sin grupo propio:", sorted(set(range(10)) - vistos))
print(f"si cada grupo se etiqueta con su dígito mayoritario: "
      f"{aciertos / len(y):.1%} de acierto")

## 3. Pocas etiquetas: autoentrenamiento (1.4)

In [ ]:
import numpy as np
import keras
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
X = x_ent[:10000].reshape(10000, -1) / 255
Xp = x_pru.reshape(10000, -1) / 255
pca = PCA(50, random_state=0).fit(X)
X, Xp, y = pca.transform(X), pca.transform(Xp), y_ent[:10000]

rng = np.random.default_rng(0)
con = np.concatenate([rng.choice(np.flatnonzero(y == d), 10,
                      replace=False) for d in range(10)])
y_semi = np.full(len(y), -1)                # -1 = sin etiqueta
y_semi[con] = y[con]
log = lambda: LogisticRegression(max_iter=2000)

solo = log().fit(X[con], y[con])
semi = SelfTrainingClassifier(log(), threshold=0.9).fit(X, y_semi)
todo = log().fit(X, y)
print("100 etiquetas:", round(solo.score(Xp, y_pru), 3))
print("100 etiquetas + 9900 sin etiqueta:",
      round(semi.score(Xp, y_pru), 3),
      "| etiquetadas por el modelo:", (semi.labeled_iter_ > 0).sum())
nuevas = semi.labeled_iter_ > 0
print("etiquetas puestas correctas:",
      round((semi.transduction_[nuevas] == y[nuevas]).mean(), 3))
print("10000 etiquetas:", round(todo.score(Xp, y_pru), 3))

## 4. Comprobar duplicados (1.5)

In [ ]:
import hashlib
from collections import Counter
import keras

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
huella = lambda img: hashlib.md5(img.tobytes()).hexdigest()
h_ent = [huella(i) for i in x_ent]
h_pru = [huella(i) for i in x_pru]
repetidas = sum(c - 1 for c in Counter(h_ent).values() if c > 1)
comunes = set(h_ent) & set(h_pru)
print("imágenes repetidas dentro de entrenamiento:", repetidas)
print("imágenes de prueba idénticas a una de entrenamiento:",
      sum(h in comunes for h in h_pru))

## 5. ¿Quién aprovecha más los datos? (1.5)

Tarda unos minutos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from sklearn.linear_model import LogisticRegression

keras.utils.set_random_seed(0)
(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
X_ent = x_ent.reshape(-1, 784) / 255.0
X_pru = x_pru.reshape(-1, 784) / 255.0

tamanos = [1000, 3000, 10_000, 30_000, 60_000]
logistica, red = [], []
for n in tamanos:
    m = LogisticRegression(max_iter=1000).fit(X_ent[:n], y_ent[:n])
    logistica.append(m.score(X_pru, y_pru))
    r = keras.Sequential([
        keras.Input(shape=(784,)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])
    r.compile("adam", "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    r.fit(X_ent[:n], y_ent[:n], epochs=5, verbose=0)
    red.append(r.evaluate(X_pru, y_pru, verbose=0)[1])
    print(f"{n:6d} imágenes: logística {logistica[-1]:.4f}, "
          f"red {red[-1]:.4f}")

plt.plot(tamanos, logistica, "o-", label="regresión logística")
plt.plot(tamanos, red, "o-", label="red con 128 neuronas ocultas")
plt.xscale("log")
plt.xlabel("imágenes de entrenamiento (escala logarítmica)")
plt.ylabel("acierto en prueba")
plt.legend()
plt.show()  # plt.savefig("mas_datos.pdf")

## 6. Aprender por bloques (1.5)

In [ ]:
import numpy as np
import keras
from sklearn.linear_model import SGDClassifier

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
np.save("x_ent.npy", x_ent.reshape(60000, -1))
X = np.load("x_ent.npy", mmap_mode="r")   # no se carga en memoria
Xp = x_pru.reshape(10000, -1) / 255

modelo = SGDClassifier(loss="log_loss", random_state=0)
bloque = 5000
for pasada in range(3):
    for i in range(0, len(X), bloque):
        trozo = np.asarray(X[i:i + bloque], dtype=np.float32) / 255
        modelo.partial_fit(trozo, y_ent[i:i + bloque],
                           classes=np.arange(10))
    print(f"pasada {pasada + 1}: acierto en prueba "
          f"{modelo.score(Xp, y_pru):.4f}")
print(f"memoria de un bloque: {trozo.nbytes / 1e6:.1f} MB; "
      f"de todo el conjunto: {X.size * 4 / 1e6:.1f} MB")

## 7. Explorar MNIST (1.6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
print("entrenamiento:", x_ent.shape, x_ent.dtype,
      " prueba:", x_pru.shape)
print("valores de los píxeles:", x_ent.min(), "a", x_ent.max())
print("imágenes por dígito:", np.bincount(y_ent))
print(f"píxeles con tinta: {(x_ent > 0).mean():.1%}")
nunca = (x_ent.max(axis=0) == 0).sum()
print("píxeles siempre en blanco:", nunca, "de 784")

fig, ejes = plt.subplots(2, 10, figsize=(4.4, 1.0))
for d in range(10):
    ejes[0, d].imshow(x_ent[np.flatnonzero(y_ent == d)[0]])
    ejes[1, d].imshow(x_ent[y_ent == d].mean(axis=0))
    ejes[0, d].set_title(str(d), fontsize=7, loc="center")
for eje in ejes.flat:
    eje.axis("off")
plt.show()  # fig.savefig("mnist.pdf")

## 8. Los vecinos más próximos (1.6)

In [ ]:
import time
import numpy as np
import keras
from sklearn.neighbors import KNeighborsClassifier

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
X = x_ent.reshape(60000, -1).astype(np.float32) / 255
Xp = x_pru.reshape(10000, -1).astype(np.float32) / 255
for k in (1, 3):
    t = time.perf_counter()
    knn = KNeighborsClassifier(k).fit(X, y_ent)
    acierto = knn.score(Xp, y_pru)
    print(f"{k} vecino(s): acierto {acierto:.4f} "
          f"({time.perf_counter() - t:.0f} s, sin entrenar nada)")
print(f"memoria del modelo: {X.nbytes / 1e6:.0f} MB "
      "(guarda todas las imágenes)")

## 9. MNIST en un plano (1.6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from sklearn.decomposition import PCA

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
pca = PCA(2).fit(x_ent.reshape(60000, -1) / 255)
print("varianza que recogen 2 componentes:",
      pca.explained_variance_ratio_.sum().round(3))
P = pca.transform(x_pru[:3000].reshape(3000, -1) / 255)
y = y_pru[:3000]
plt.figure(figsize=(4.4, 2.8))
for d in range(10):
    plt.scatter(*P[y == d].T, s=2, color=plt.cm.tab10(d), label=d)
    plt.annotate(str(d), P[y == d].mean(axis=0), weight="bold",
                 fontsize=9, ha="center")
plt.xlabel("componente 1"); plt.ylabel("componente 2")
plt.show()  # plt.savefig("plano.pdf")
centros = np.array([P[y == d].mean(axis=0) for d in range(10)])
d01 = np.linalg.norm(centros[0] - centros[1])
d49 = np.linalg.norm(centros[4] - centros[9])
print(f"distancia entre centros: 0-1 {d01:.2f}, 4-9 {d49:.2f}")

## 10. Regresión logística: el modelo de referencia (1.7)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
from sklearn.linear_model import LogisticRegression

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
X_ent = x_ent.reshape(-1, 784) / 255.0   # 784 variables entre 0 y 1
X_pru = x_pru.reshape(-1, 784) / 255.0

# 1) modelo estadístico: ¿qué píxeles distinguen un 0 de un 1?
sel = y_ent <= 1
m01 = LogisticRegression(max_iter=1000).fit(X_ent[sel], y_ent[sel])
sel_p = y_pru <= 1
print(f"0 frente a 1: {m01.score(X_pru[sel_p], y_pru[sel_p]):.4f}")
plt.figure(figsize=(2.6, 2.2))
plt.imshow(m01.coef_.reshape(28, 28), cmap="RdBu_r",
           vmin=-1, vmax=1)
plt.colorbar(label="coeficiente (+: más probable un 1)")
plt.axis("off")
plt.show()  # plt.savefig("coeficientes_01.pdf")

# 2) modelo de referencia para los diez dígitos
t0 = time.perf_counter()
m = LogisticRegression(max_iter=1000).fit(X_ent, y_ent)
print(f"10 dígitos: {m.score(X_pru, y_pru):.4f} de acierto en prueba"
      f" ({time.perf_counter() - t0:.0f} s, "
      f"{m.coef_.size + m.intercept_.size} parámetros)")

## 11. La ley de Engel (1.7)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

d = sm.datasets.engel.load_pandas().data     # 235 hogares belgas
X = sm.add_constant(d.income)
lin = sm.OLS(d.foodexp, X).fit()
print(lin.params.round(3).to_dict(), "R2", round(lin.rsquared, 3))
print("IC 95 % de la pendiente:",
      lin.conf_int().loc["income"].round(3).tolist())
print("Breusch-Pagan, p =",
      f"{het_breuschpagan(lin.resid, X)[1]:.1e}")

Xl = sm.add_constant(np.log(d.income))
log = sm.OLS(np.log(d.foodexp), Xl).fit()
print("elasticidad:", round(log.params.iloc[1], 3),
      "IC 95 %:", log.conf_int().iloc[1].round(3).tolist(),
      "R2", round(log.rsquared, 3))
print("Breusch-Pagan (log), p =",
      f"{het_breuschpagan(log.resid, Xl)[1]:.1e}")
rob = sm.OLS(d.foodexp, X).fit(cov_type="HC3")  # errores robustos
print("IC 95 % robusto de la pendiente:",
      rob.conf_int().loc["income"].round(3).tolist())

fig, (a, b) = plt.subplots(1, 2, figsize=(4.4, 1.8),
                           layout="constrained")
a.scatter(d.income, d.foodexp, s=4)
xs = np.linspace(d.income.min(), d.income.max(), 2)
a.plot(xs, lin.params.iloc[0] + lin.params.iloc[1] * xs)
a.set_xlabel("ingresos"); a.set_ylabel("gasto en comida")
a.set_title("A. Recta ajustada")
b.scatter(lin.fittedvalues, lin.resid, s=4)
b.axhline(0, color="0.5", lw=0.8)
b.set_xlabel("valor ajustado"); b.set_title("B. Residuos")
plt.show()  # fig.savefig("engel.pdf")

**Ejercicio.** Ajusta la regresión del gasto frente a los ingresos solo con los hogares de ingresos inferiores a 2000. ¿Cambia la pendiente?

> Koenker, R. y Bassett, G. (1982). Robust tests for heteroscedasticity based on regression quantiles. *Econometrica*, 50(1), 43–61. https://doi.org/10.2307/1912528
>
> Zhu, X. y Goldberg, A. B. (2009). *Introduction to Semi-Supervised Learning*. Morgan & Claypool. ISBN 978-1-59829-547-4